# 第 10 章 パネルデータ: 数字を変えて確かめる

**使い方**: 「Parameters」のセルの数字を書き換えて、上から順に実行します（メニューの Run → Run All Cells）。数字を変えるたびに、結果と図が変わります。

起動: アプリのディレクトリで `uv run --with jupyterlab jupyter lab notebooks/` を実行します。

記事: `docs/article/econometrics-programming/10-panel-data.md`

In [ ]:
# --- Parameters: ここの数字を書き換えます ---
seed = 0             # 乱数の種
entities = 100       # 個人の数
periods = 5          # 期間の数
correlation = 1.0    # 個人の効果 a と x の相関の強さ（0 なら無相関）
repetitions = 100    # パネルを作り直す回数

In [ ]:
# --- 準備（変更しません）: lib を読み込めるようにします ---
import sys
from pathlib import Path

_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "lib").is_dir())
sys.path[:] = [p for p in sys.path if p != str(_root)]
sys.path.insert(0, str(_root))
_lib = sys.modules.get("lib")
if _lib is not None and (
    Path(getattr(_lib, "__file__", "") or "").resolve()
    != (_root / "lib" / "__init__.py").resolve()
):
    for _name in list(sys.modules):
        if _name == "lib" or _name.startswith("lib."):
            del sys.modules[_name]

## 1 つのパネルを、固定効果とランダム効果で推定する

In [ ]:
import numpy as np

from lib.chapter09.did import draw_panel, pooled_estimate
from lib.chapter10.panel import fixed_effects, hausman_test, random_effects

panel = draw_panel(entities, periods, correlation, np.random.default_rng(seed))
fe = fixed_effects(panel, "y", ["x"])
re = random_effects(panel, "y", ["x"])
print(f"プーリング OLS {pooled_estimate(panel):.3f}、"
      f"固定効果 {fe.coefficients['x']:.3f}、ランダム効果 {re.coefficients['x']:.3f}"
      "（真の傾きは 1）")
test = hausman_test(fe, re, ["x"])
print(f"Hausman 検定: 統計量 {test.statistic:.3f}、p 値 {test.p_value:.4f}")

## 3 つの推定量の分布（パネルを作り直して繰り返す）

In [ ]:
from lib.chapter10.panel import estimator_comparison, hausman_rejection_rate

estimates = estimator_comparison(
    entities, periods, correlation, repetitions, np.random.default_rng(seed)
)
print(estimates.agg(["mean", "std"]).round(3))
rate = hausman_rejection_rate(
    entities, periods, correlation, repetitions, 0.05, np.random.default_rng(seed)
)
print(f"Hausman 検定の棄却率 {rate:.3f}")

## 図: 推定量ごとの分布

In [ ]:
from lib.chapter10.plot import plot_estimate_comparison

plot_estimate_comparison(estimates, 1.0)

## 図: 個人ごとの直線

In [ ]:
from lib.chapter10.plot import plot_entity_lines

plot_entity_lines(panel, 5)

## 試してみよう

- `correlation` を 0.0 にすると、プーリング OLS・固定効果・ランダム効果が、どれも 1
  の近く（0.970・0.989・0.988）になり、Hausman 検定の p 値は 0.6669 で、
  同じという帰無仮説を棄却しません
- `correlation` が 1.0 のときは、固定効果が 0.989、ランダム効果が 1.177 で、Hausman
  検定の p 値は 0.0000 です。個人の効果が x と相関すると、ランダム効果は偏ります
- 3 つの推定量を繰り返した結果（`repetitions` が 100）は、固定効果 0.998（標準偏差
  0.022）、ランダム効果 1.213、プーリング OLS 1.491 です